# Submits 33 a 37 — ensamble de 100 modelos con variables nuevas (BC_limpio), código completo de principio a fin

Igual que el submit 32 (entrega final hasta ahora) **más las variables nuevas "BC_limpio"** de z026/z027
(`0_exploracion/z027_bc_limpio.ipynb`). Un solo cambio respecto del 32: las variables.

**Receta**
1. Datos: `competencia_01.parquet` (clase_ternaria recalculada; `ccajas_depositos` = NA en 202105).
2. Variables: las 761 del submit 32 (originales + lags y deltas de orden 1 y 2) + las nuevas de BC_limpio
   (creadas antes de los lags; sus lags/deltas son solo los de la lista de z027).
3. Target juntar (BAJA+1 o BAJA+2 = 1). Train: 202103 a 202106 completos, sin undersampling.
4. Modelos: 10 mejores trials de z005 × 10 semillas (las del 32, en los mismos dos grupos) = **100 modelos**.
5. Ensamble: promedio simple de las 100 probabilidades por cliente de 202108 → un archivo por corte:

| Submit | Archivo | Corte |
|---|---|---|
| 33 | `33_ens100juntarBC_12000.csv` | top 12.000 |
| 34 | `34_ens100juntarBC_11500.csv` | top 11.500 |
| 35 | `35_ens100juntarBC_11000.csv` | top 11.000 |
| 36 | `36_ens100juntarBC_10500.csv` | top 10.500 |
| 37 | `37_ens100juntarBC_12500.csv` | top 12.500 |

**Tiempo:** ~4 h. Cada predicción se guarda en `probs_33/`: si se corta, volver a correr y retoma.

In [1]:
# --- 1. Librerías y parámetros ---
import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

MESES_FINAL = [202103, 202104, 202105, 202106]   # meses de entrenamiento
MES_PRED    = 202108                             # mes a predecir
VERSION     = "BC_limpio"                               # variables nuevas que se agregan (z027)
ENVIOS      = {33: 12000, 34: 11500, 35: 11000, 36: 10500, 37: 12500}                                 # número de submit -> cantidad de clientes
CARPETA     = "probs_33"                         # una predicción por modelo (para retomar si se corta)
os.makedirs(CARPETA, exist_ok=True)

# Dos grupos de 5 semillas (los del submit 32). Cada grupo usa su propio Dataset (ver celda 4).
GRUPOS_SEMILLAS = {
    "submit15": [501419, 623353, 786349, 834569, 946331],
    "submit16": [270131, 417293, 583117, 692347, 851071],
}

# Hiperparámetros fijos (recomendación del profesor)
params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}

# Los 10 mejores trials de z005: trial -> (num_leaves, min_sum_hessian_in_leaf, cantidad de árboles)
TRIALS = {
    34: (137, 9.755574121446362, 538),
    36: (61, 7.745373665319586, 727),
    27: (241, 5.632060378965947, 578),
    29: (193, 5.9684882171070806, 567),
    37: (240, 0.0030389429440898, 888),
    31: (203, 1.4449699818050468, 532),
    20: (159, 0.0215427682411664, 843),
    9:  (65, 1.2505922499454127, 684),
    28: (237, 0.0956720722437809, 826),
    39: (118, 1.942966740643147, 664),
}

In [2]:
# --- 2. Datos, ordenados por cliente y mes (para que shift(k) traiga el valor de k meses antes del MISMO cliente) ---
df = pd.read_parquet("../competencia_01.parquet")
no_usar = ["foto_mes", "clase_ternaria"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]   # a estas se les calculan lags
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)

In [3]:
# --- Variables nuevas de z026 (versión "limpia"), ANTES de los lags. df tiene que estar ordenado por cliente y mes ---
def s(*columnas):
    # Suma con fillna(0) en cada término: si no tiene Master, sus columnas son NaN y la suma daría NaN
    total = 0
    for c in columnas:
        total = total + (df[c].fillna(0) if isinstance(c, str) else c.fillna(0))
    return total

def div(a, b):
    # Cociente: si el denominador es 0 o negativo, NaN. Ej.: 1.000 dólares / saldo −5.000 → NaN (no tiene sentido)
    return (a / b.where(b > 0)).astype("float32")

n = {}
# Auxiliares de tarjetas (las usa el grupo C; el grupo A no entra)
n["c_tarjetas_trx"]      = s("ctarjeta_visa_transacciones", "ctarjeta_master_transacciones")

# ---------- Grupo B: plata del cliente (sin c_cheques_rechazados, gain 0 en z026) ----------
n["m_saldo_paquete"]     = s("mcuenta_corriente", "mcaja_ahorro")
n["m_saldo_adicional"]   = s("mcuenta_corriente_adicional", "mcaja_ahorro_adicional")
n["r_saldo_dolares"]     = div(df["mcaja_ahorro_dolares"], df["mcuentas_saldo"])
n["m_inversiones_total"] = s("mplazo_fijo_pesos", "mplazo_fijo_dolares", "minversion1_pesos", "minversion1_dolares", "minversion2")
n["m_patrimonio_banco"]  = s("mcuentas_saldo", n["m_inversiones_total"])
n["m_forex_neto"]        = df["mforex_buy"].fillna(0) - df["mforex_sell"].fillna(0)
n["m_payroll_total"]     = s("mpayroll", "mpayroll2")
n["m_ingresos_total"]    = s(n["m_payroll_total"], "mtransferencias_recibidas", "mcheques_depositados")
n["m_egresos_total"]     = s("mtransferencias_emitidas", "mcheques_emitidos", "mextraccion_autoservicio",
                             "mpagodeservicios", "mpagomiscuentas", "mcuenta_debitos_automaticos")
n["m_flujo_neto"]        = n["m_ingresos_total"] - n["m_egresos_total"]
n["m_debitos_aut_total"] = s("mcuenta_debitos_automaticos", "mttarjeta_visa_debitos_automaticos", "mttarjeta_master_debitos_automaticos")
n["m_descuentos_total"]  = s("mcajeros_propios_descuentos", "mtarjeta_visa_descuentos", "mtarjeta_master_descuentos")

# ---------- Grupo C: uso y fidelidad ----------
n["c_inversiones_total"]  = s("cplazo_fijo", "cinversion1", "cinversion2")
n["c_seguros_total"]      = s("cseguro_vida", "cseguro_auto", "cseguro_vivienda", "cseguro_accidentes_personales")
n["c_payroll_total"]      = s("cpayroll_trx", "cpayroll2_trx")
n["tiene_payroll"]        = (n["c_payroll_total"] > 0).astype("int8")
n["c_debitos_aut_total"]  = s("ccuenta_debitos_automaticos", "ctarjeta_visa_debitos_automaticos", "ctarjeta_master_debitos_automaticos")
n["c_pagos_servicios_total"] = s("cpagodeservicios", "cpagomiscuentas", n["c_debitos_aut_total"])
n["c_descuentos_total"]   = s("ccajeros_propios_descuentos", "ctarjeta_visa_descuentos", "ctarjeta_master_descuentos")
n["c_comisiones_total"]   = s("ccomisiones_mantenimiento", "ccomisiones_otras")
n["c_trx_digitales"]      = s("chomebanking_transacciones", "cmobile_app_trx")     # la app vale 0/1: suma 1 si la usó
n["c_trx_presenciales"]   = s("ccajas_transacciones", "catm_trx", "catm_trx_other")
n["r_atm_otros"]          = div(df["catm_trx_other"].fillna(0), s("catm_trx", "catm_trx_other"))
n["c_trx_total"]          = s(n["c_trx_digitales"], n["c_trx_presenciales"], "ctarjeta_debito_transacciones",
                              n["c_tarjetas_trx"], "ccallcenter_transacciones")   # el call center cuenta como actividad
usados = [df[c] for c in ["ctarjeta_debito_transacciones", "cpagodeservicios", "cpagomiscuentas", "chomebanking_transacciones",
                          "cmobile_app_trx", "catm_trx", "ccajas_transacciones", "ctransferencias_emitidas",
                          "ctransferencias_recibidas", "cforex"]] + \
         [n[c] for c in ["c_tarjetas_trx", "c_payroll_total", "c_debitos_aut_total", "m_inversiones_total", "c_seguros_total"]]
n["c_servicios_usados"]   = sum((x > 0).astype("int8") for x in usados)            # 0 a 15 (NaN cuenta como no usado)
n["r_uso_vs_tenencia"]    = div(n["c_servicios_usados"].astype("float32"), df["cproductos"])
n["banco_principal"]      = ((n["tiene_payroll"] == 1) & (n["c_debitos_aut_total"] > 0) & (n["c_tarjetas_trx"] > 0)).astype("int8")

cliente = df["numero_de_cliente"]
tp = n["tiene_payroll"]
n["meses_con_payroll_3"] = (tp + tp.groupby(cliente).shift(1).fillna(0) + tp.groupby(cliente).shift(2).fillna(0)).astype("int8")
# Racha de meses seguidos activo: cada mes inactivo abre un "bloque" nuevo; dentro del bloque se suman los meses activos
activo = (n["c_servicios_usados"] > 0).astype("int16")
bloque = (activo == 0).groupby(cliente).cumsum()
n["meses_activo_seguidos"] = activo.groupby([cliente, bloque]).cumsum().astype("int8")

nuevas_df = pd.DataFrame(n).astype("float32")
del n, usados

GRUPO_B = ["m_saldo_paquete", "m_saldo_adicional", "r_saldo_dolares", "m_patrimonio_banco", "m_inversiones_total", "m_forex_neto",
           "m_payroll_total", "m_ingresos_total", "m_egresos_total", "m_flujo_neto", "m_debitos_aut_total", "m_descuentos_total"]
GRUPO_C = ["c_inversiones_total", "c_seguros_total", "c_payroll_total", "tiene_payroll", "c_debitos_aut_total",
           "c_pagos_servicios_total", "c_descuentos_total", "c_comisiones_total", "c_trx_digitales", "c_trx_presenciales",
           "r_atm_otros", "c_trx_total", "c_servicios_usados", "r_uso_vs_tenencia", "banco_principal",
           "meses_con_payroll_3", "meses_activo_seguidos"]
M_CON_LAG1   = [c for c in GRUPO_B if c.startswith("m_")]                  # 11 montos: lag 1 y delta 1 (sin lag2/delta2: gain 0)
C_CON_DELTA1 = ["c_trx_total", "c_servicios_usados", "c_trx_digitales"]    # solo delta 1
g_nuevas = nuevas_df.groupby(df["numero_de_cliente"])
historia = {}
lag_m = g_nuevas[M_CON_LAG1].shift(1)
for c in M_CON_LAG1:
    historia[f"{c}_lag1"] = lag_m[c]
    historia[f"{c}_delta1"] = nuevas_df[c] - lag_m[c]
lag_c = g_nuevas[C_CON_DELTA1].shift(1)
for c in C_CON_DELTA1:
    historia[f"{c}_delta1"] = nuevas_df[c] - lag_c[c]
historia = pd.DataFrame(historia).astype("float32")

C_LIMPIO  = GRUPO_C + [f"{c}_delta1" for c in C_CON_DELTA1]                                      # 20 variables
B_LIMPIO  = GRUPO_B + [f"{c}_{t}1" for c in M_CON_LAG1 for t in ["lag", "delta"]]                 # 34 variables
BC_LIMPIO = B_LIMPIO + C_LIMPIO                                                                    # 54 variables
nuevas_df = pd.concat([nuevas_df, historia], axis=1)[BC_LIMPIO]       # me quedo solo con las que se usan
del historia, g_nuevas, lag_m, lag_c
print(f"C limpio: {len(C_LIMPIO)}  B+C limpio: {len(BC_LIMPIO)}")

C limpio: 20  B+C limpio: 54


In [4]:
# --- 4. Lags/deltas de orden 1 y 2 de las originales (como en el 32), unir las nuevas y armar train / 202108 ---
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")              # valor de hace k meses
    delta = (df[cols] - lag).astype("float32")          # cuánto cambió desde entonces
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

base = [c for c in df.columns if c not in no_usar]        # las 761 del submit 32
extra = {"C_limpio": C_LIMPIO, "BC_limpio": BC_LIMPIO}[VERSION]
df = pd.concat([df, nuevas_df[extra]], axis=1)
del nuevas_df
variables = base + extra

final  = df[df["foto_mes"].isin(MESES_FINAL)]
y_final = final["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)   # target juntar
futuro = df[df["foto_mes"] == MES_PRED][["numero_de_cliente"]].copy()
X_futuro = df.loc[futuro.index, variables]
print("Variables:", len(variables), f"(761 + {len(extra)} nuevas)", "| filas train:", len(final), "| positivos:", y_final.sum())

Variables: 815 (761 + 54 nuevas) | filas train: 654066 | positivos: 8067


In [5]:
# --- 5. Funciones (igual que el 32) ---
def armar_dataset(semilla_grupo):
    # Los bines (max_bin = 31) se arman con una muestra elegida con la semilla: un Dataset por grupo, con su primera semilla
    datos = lgb.Dataset(final[variables], label=y_final, free_raw_data=False,
                        params={**params_fijos, "seed": semilla_grupo})
    return datos.construct()

def entrenar_y_predecir(datos, trial, semilla):
    hojas, hessian, arboles = TRIALS[trial]
    p = {**params_fijos, "num_leaves": hojas, "min_sum_hessian_in_leaf": hessian, "seed": semilla}
    modelo = lgb.train(p, datos, num_boost_round=arboles)
    return modelo.predict(X_futuro)                  # probabilidad de baja de cada cliente de 202108

In [6]:
# --- 6. Entrenar los 100 modelos (retoma si se cortó) ---
inicio_total = time.time()
for nombre, semillas in GRUPOS_SEMILLAS.items():
    if all(os.path.exists(f"{CARPETA}/t{t}_s{s}.npy") for t in TRIALS for s in semillas):
        continue
    datos = armar_dataset(semillas[0])
    for trial in TRIALS:
        for semilla in semillas:
            archivo = f"{CARPETA}/t{trial}_s{semilla}.npy"
            if os.path.exists(archivo):
                continue
            inicio = time.time()
            np.save(archivo, entrenar_y_predecir(datos, trial, semilla))
            with open("33_progreso.txt", "a", encoding="utf-8") as f:
                f.write(time.strftime("%H:%M:%S ") + f"{nombre} | trial {trial}, semilla {semilla}: {(time.time() - inicio) / 60:.1f} min\n")
    del datos
print(f"Total: {(time.time() - inicio_total) / 60:.0f} min")

Total: 266 min


In [7]:
# --- 7. Ensamble (promedio de las 100 probabilidades) y un archivo por corte ---
todas = [s for semillas in GRUPOS_SEMILLAS.values() for s in semillas]
probs = [np.load(f"{CARPETA}/t{t}_s{s}.npy") for t in TRIALS for s in todas]
futuro["prob_ensamble"] = np.mean(probs, axis=0)
print("Modelos en el ensamble:", len(probs))
futuro.to_parquet("33_ens100_probs.parquet")
orden = futuro.sort_values("prob_ensamble", ascending=False)["numero_de_cliente"].astype("int64")

etiqueta = {"C_limpio": "C", "BC_limpio": "BC"}[VERSION]
entrega_32 = set(pd.read_csv("32_ens100juntar_12000.csv", header=None)[0])
for numero, n in ENVIOS.items():
    archivo = f"{numero}_ens100juntar{etiqueta}_{n}.csv"
    top = orden.head(n)
    top.to_csv(archivo, index=False, header=False)     # una columna, sin encabezado (formato del bot)
    contenido = open(archivo).read()
    print(f"{archivo} | filas: {len(contenido.splitlines())} | únicos: {top.nunique()} | ¿e+? {'e+' in contenido}"
          f" | coincide con el 32: {len(entrega_32 & set(top)) / len(entrega_32):.1%}")

Modelos en el ensamble: 100


33_ens100juntarBC_12000.csv | filas: 12000 | únicos: 12000 | ¿e+? False | coincide con el 32: 96.6%
34_ens100juntarBC_11500.csv | filas: 11500 | únicos: 11500 | ¿e+? False | coincide con el 32: 94.2%
35_ens100juntarBC_11000.csv | filas: 11000 | únicos: 11000 | ¿e+? False | coincide con el 32: 90.8%


36_ens100juntarBC_10500.csv | filas: 10500 | únicos: 10500 | ¿e+? False | coincide con el 32: 87.2%
37_ens100juntarBC_12500.csv | filas: 12500 | únicos: 12500 | ¿e+? False | coincide con el 32: 98.0%


## Resultados en el bot

(se completa cuando se envíen)